In [ ]:
"""
Бизнес-контекст:
Банк теряет клиентов. Каждый ушедший клиент — это не только потерянная будущая прибыль, но и затраты на 
привлечение нового (в 5-25 раз дороже, чем удержание старого). Отдел маркетинга просит вас построить модель, 
которая по поведенческим и демографическим данным клиентов — владельцев кредитных карт — сможет предсказать 
вероятность оттока в ближайшее время. На основе предсказаний банк сможет запустить таргетированную кампанию: предложить 
удерживающие бонусы (кешбэк, снижение ставки) тем, кто с высокой вероятностью уйдёт.

Что нужно сделать:

EDA и аналитика: исследовать данные (распределения, корреляции, сравнение групп «ушёл / остался», построить визуализации).

Предобработка и Feature Engineering: удалить признаки-«утечки», создать новые признаки 
(например, inactivity_score = inactive_months × (1 / contact_count)), закодировать категории, отмасштабировать числа, сбалансировать классы (SMOTE).

Обучение моделей: обучить логистическую регрессию (базовый уровень), Random Forest (интерпретируемая) и LightGBM (лучшее качество для табличных данных).

Оценка качества: использовать ROC-AUC (главная метрика для несбалансированных данных), Precision/Recall, 
построить confusion matrix и матрицу ошибок по сегментам (например, для разных возрастных групп).

Интерпретация: построить feature importance (модель Random Forest или SHAP), определить топ-5 факторов оттока.

Бизнес-рекомендации: на основе полученных инсайтов предложить, на каких клиентов и как воздействовать, и оценить потенциальный эффект.
"""

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import lightgbm as lgb
import shap

# -------------------- 1. ЗАГРУЗКА И ПЕРВИЧНЫЙ ОСМОТР --------------------
# Ссылка на датасет: https://www.kaggle.com/datasets/sakshigoyal7/credit-card-customers
df = pd.read_csv('BankChurners.csv')

print("Размер данных:", df.shape)
print("\nПервые 5 строк:")
print(df.head())

# TODO 1.1: Выведите информацию о типах данных и пропусках (.info(), .isnull().sum())
# TODO 1.2: Посчитайте распределение целевой переменной Attrition_Flag
#   (доля Existing Customer vs Attrited Customer)
#   Сделайте вывод: сбалансирована ли задача?
# TODO 1.3: Удалите признаки-«утечки» (которые содержат информацию о будущем)
#   Обычно это: 'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_...',
#   и любые другие, где явно видно подглядывание.

# -------------------- 2. РАЗВЕДОЧНЫЙ АНАЛИЗ (EDA) --------------------
# TODO 2.1: Постройте распределение числовых признаков (гистограммы с groupby по Attrition_Flag)
#   Особое внимание: Customer_Age, Credit_Limit, Total_Trans_Amt, Total_Revolving_Bal,
#   Months_Inactive_12_mon, Contacts_Count_12_mon.
#   Используйте sns.histplot с параметром hue='Attrition_Flag'
# TODO 2.2: Постройте boxplot зависимости числовых признаков от Attrition_Flag
#   Какие признаки визуально разделяют группы?
# TODO 2.3: Постройте матрицу корреляций для числовых признаков (тепловая карта)
#   Найдите признаки, которые сильно коррелируют между собой (мультиколлинеарность)
# TODO 2.4: Для категориальных признаков (Education_Level, Marital_Status, Card_Category)
#   постройте countplot (stacked bars) с группировкой по Attrition_Flag.
#   Напишите текстовый вывод: какие категории более склонны к оттоку?

# -------------------- 3. ПРЕДОБРАБОТКА ДАННЫХ --------------------
# Определите числовые и категориальные столбцы
num_features = ['Customer_Age', 'Dependent_count', 'Months_on_book',
                'Total_Relationship_Count', 'Months_Inactive_12_mon',
                'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal',
                'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt',
                'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio']

cat_features = ['Gender', 'Education_Level', 'Marital_Status', 'Income_Category',
                'Card_Category']

# TODO 3.1: Создайте новый признак Inactivity_Score = Months_Inactive_12_mon * (1 / (Contacts_Count_12_mon + 1))
#   Это агрегированный поведенческий сигнал: чем больше месяцев бездействия и чем реже контакты,
#   тем выше риск оттока.
# TODO 3.2: Добавьте этот признак в num_features.

# TODO 3.3: Создайте ColumnTransformer:
#   - для числовых: StandardScaler()
#   - для категориальных: OneHotEncoder(drop='first', handle_unknown='ignore')
# TODO 3.4: Разделите данные на X и y (целевая переменная).
#   Преобразуйте Attrition_Flag в бинарную: 1 — Attrited Customer, 0 — Existing Customer.
# TODO 3.5: Разбейте данные на train/test (80/20) с сохранением стратификации (stratify=y)

# -------------------- 4. МОДЕЛИРОВАНИЕ (3 модели) --------------------
# Для борьбы с дисбалансом классов (примерно 84%/16% в пользу Existing Customer)
# используем SMOTE только на тренировочных данных.

models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'LightGBM': lgb.LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)
}

results = {}

for name, model in models.items():
    print(f"\n=== Training {name} ===")
    pipeline = ImbPipeline([
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('classifier', model)
    ])
    pipeline.fit(X_train, y_train)
    y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_pred_proba)
    results[name] = {
        'pipeline': pipeline,
        'proba': y_pred_proba,
        'auc': auc
    }
    print(f"AUC-ROC: {auc:.4f}")

# TODO 4.1: Для модели RandomForest дополнительно выведите feature importance
#   (имена признаков после preprocessor). Постройте горизонтальный barplot топ-10.

# -------------------- 5. ОЦЕНКА КАЧЕСТВА --------------------
# TODO 5.1: Постройте ROC-кривые для всех трёх моделей на одном графике.
# TODO 5.2: Для лучшей модели (по AUC) постройте confusion matrix (нормализованную и абсолютную).
# TODO 5.3: Для лучшей модели выведите classification_report (precision, recall, f1-score).
# TODO 5.4: Проанализируйте, где модель ошибается: какие клиенты (по признакам) чаще получают
#   ложноположительные или ложноотрицательные ответы.

# -------------------- 6. ИНТЕРПРЕТАЦИЯ С SHAP --------------------
# Выберите лучшую модель (например, LightGBM или RandomForest)
best_model = results['LightGBM']['pipeline']

# Получим трансформированные тестовые данные (без SMOTE)
X_test_transformed = preprocessor.transform(X_test)
feature_names = (preprocessor.named_transformers_['num'].feature_names_in_.tolist() +
                 list(preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(cat_features)))

explainer = shap.TreeExplainer(best_model.named_steps['classifier'])
shap_values = explainer.shap_values(X_test_transformed)

# TODO 6.1: Постройте SHAP summary plot (глобальная важность признаков)
# TODO 6.2: Выберите двух конкретных клиентов из теста (один ушедший, один лояльный)
#   и постройте SHAP waterfall plot для объяснения предсказания.

# -------------------- 7. ВЫВОДЫ И БИЗНЕС-РЕКОМЕНДАЦИИ --------------------
"""
Напишите здесь развёрнутые выводы и рекомендации, отвечая на вопросы:

1. Какая модель показала лучшее качество и почему (опираясь на ROC-AUC и другие метрики)?
2. Какие топ-5 факторов оттока вы выявили? Как банк может повлиять на эти факторы?
3. Какие профили клиентов (возраст, доход, категория карты, количество контактов и т.д.) оказались наиболее рискованными?
4. Что можно предложить банку в качестве удерживающих мероприятий? Оцените ожидаемый эффект (хотя бы качественно).
5. Какие ограничения у вашей модели и что можно было бы улучшить при наличии большего времени/данных?
"""

# -------------------- Дополнительное задание (по желанию) --------------------
# 1. Подберите гиперпараметры для LightGBM с помощью Optuna или GridSearchCV.
# 2. Постройте калибровочную кривую (calibration curve) для лучшей модели.
# 3. Используйте StratifiedKFold для кросс-валидации и усреднения метрики.